# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [4]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses *(À compléter)* :**
- **Test #1 Khi²:**
  - $H_0 =$ `la gravité de l'accident est **indépendante** des conditions de luminosité` / $H_1 =$ `la gravité de l'accident dépend des conditions de luminosité`
  - *Justification du choix du test :* `les deux variables sont qualitatives (luminosité : jour / nuit éclairée / nuit sans éclairage… ; gravité : Slight / Serious / Fatal). Le test #1 d'indépendance est le test adapté pour mesurer le lien entre deux variables qualitatives.`
- **Test #2 Mann-Whitney:**
  - $H_0 =$ `le nombre de victimes par accident a la même distribution en zone rurale et en zone urbaine` / $H_1 =$ `le nombre de victimes par accident est plus élevé en zone rurale qu'en zone urbaine`
  - Justification du choix du test :* `on compare 2 groupes (rural et urbain). Le nombre de victimes est très déséquilibré : presque toujours 1 victime, mais parfois beaucoup plus (jusqu'à 93). Mann-Whitney n'est pas faussé par ces valeurs extrêmes, contrairement à un test basé sur la moyenne.`

In [8]:
SILVER_MAIN_PATH = "/lakehouse/silver/accidents_enriched"
ALPHA = 0.05   # seuil de décision : si p < 0,05 on rejette H0

# 1) Préparer les données : projection + nettoyage + échantillon CÔTÉ SPARK
df_sample = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .select("light_conditions", "accident_severity",
            "urban_or_rural_area", "number_of_casualties")      # seulement 4 colonnes
    .dropna()                                                   # on enlève les valeurs manquantes
    .filter(F.col("urban_or_rural_area").isin("Rural", "Urban"))
    .sample(fraction=0.05, seed=42)                             # ~5 % ≈ 100 000 lignes
    .toPandas()                                                 # seulement maintenant vers pandas
)
assert df_sample is not None, "TODO : Extrayez les colonnes nécessaires dans `df_sample`."
print(f"Échantillon : {len(df_sample):,} accidents (sur 2 047 256)\n")

# 2) TEST #1 — Khi² : luminosité × gravité

print("=" * 60)
print("TEST #1 — Khi² : luminosité × gravité")
print("=" * 60)

# Tableau croisé : combien d'accidents pour chaque (luminosité, gravité)
tableau = pd.crosstab(df_sample["light_conditions"], df_sample["accident_severity"])
print("Tableau croisé (nombre d'accidents) :")
print(tableau, "\n")

# Le même tableau en % par ligne (plus lisible)
print("En % par luminosité :")
print((pd.crosstab(df_sample["light_conditions"], df_sample["accident_severity"],
                   normalize="index") * 100).round(1), "\n")

chi2, p_val_1, dof, _ = stats.chi2_contingency(tableau)

# Taille de l'effet : V de Cramér (0 = aucun lien, 1 = lien parfait)
n = tableau.to_numpy().sum()
v_cramer = np.sqrt(chi2 / (n * (min(tableau.shape) - 1)))

print(f"Khi² = {chi2:,.1f}   |   ddl = {dof}   |   p-value = {p_val_1:.2e}")
print(f"V de Cramér = {v_cramer:.3f}")
print("→ On rejette H0 : la gravité DÉPEND de la luminosité." if p_val_1 < ALPHA
      else "→ On ne peut pas rejeter H0.")

print("\n" + "=" * 320
     )

# 3) TEST #2 — Mann-Whitney : victimes rural vs urbain

print("\n" + "=" * 60)
print("TEST #2 — Mann-Whitney : nombre de victimes, rural vs urbain")
print("=" * 60)

rural  = df_sample.loc[df_sample["urban_or_rural_area"] == "Rural", "number_of_casualties"]
urbain = df_sample.loc[df_sample["urban_or_rural_area"] == "Urban", "number_of_casualties"]

# Résumé des 2 groupes
resume = pd.DataFrame({
    "nb_accidents": [len(rural), len(urbain)],
    "mediane":      [rural.median(), urbain.median()],
    "moyenne":      [round(rural.mean(), 3), round(urbain.mean(), 3)],
    "pct_2_victimes_ou_plus": [round((rural >= 2).mean() * 100, 1),
                               round((urbain >= 2).mean() * 100, 1)],
}, index=["Rural", "Urbain"])
print(resume, "\n")

# alternative="greater" : on teste précisément H1 « rural > urbain »
stat_2, p_val_2 = stats.mannwhitneyu(rural, urbain, alternative="greater")

# Taille de l'effet : probabilité qu'un accident rural ait plus de victimes qu'un urbain
proba_sup = stat_2 / (len(rural) * len(urbain))

print(f"U = {stat_2:,.0f}   |   p-value = {p_val_2:.2e}")
print(f"Probabilité de supériorité = {proba_sup:.3f}  (0,5 = aucune différence)")
print("→ On rejette H0 : il y a plus de victimes par accident en zone rurale." if p_val_2 < ALPHA
      else "→ On ne peut pas rejeter H0.")

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Échantillon : 102,462 accidents (sur 2 047 256)

TEST #1 — Khi² : luminosité × gravité
Tableau croisé (nombre d'accidents) :
accident_severity            Fatal  Serious  Slight
light_conditions                                   
Darkness - lighting unknown     20      170    1033
Darkness - lights lit          279     3019   16889
Darkness - lights unlit          7       78     394
Darkness - no lighting         238     1159    4347
Daylight                       743    10032   64054 

En % par luminosité :
accident_severity            Fatal  Serious  Slight
light_conditions                                   
Darkness - lighting unknown    1.6     13.9    84.5
Darkness - lights lit          1.4     15.0    83.7
Darkness - lights unlit        1.5     16.3    82.3
Darkness - no lighting         4.1     20.2    75.7
Daylight                       1.0     13.4    85.6 

Khi² = 679.8   |   ddl = 8   |   p-value = 1.57e-141
V de Cramér = 0.058
→ On rejette H0 : la gravité DÉPEND de la lumino

#### Interprétation des tests

**Test #1 — Luminosité × gravité (Khi²)**
- p-value = 1,57e-141, donc bien plus petite que 0,05 → **on rejette H0** : la gravité **dépend** de la luminosité.
- La nuit sans éclairage, **4,1 %** des accidents sont mortels, contre **1,0 %** le jour : **4 fois plus**.
- Le V de Cramér (0,058) est faible : la plupart des accidents restent légers, quelle que soit la luminosité. Le lien est **réel**, mais il ne concerne qu'une partie des accidents.

**Test #2 — Victimes rural vs urbain (Mann-Whitney)**
- p-value ≈ 0, donc plus petite que 0,05 → **on rejette H0** : les accidents à la campagne font **plus de victimes**.
- La médiane vaut 1 dans les deux cas, mais **30 %** des accidents ruraux font 2 victimes ou plus, contre **19 %** en ville.
- La probabilité de supériorité (0,56) indique un effet modéré : un accident rural a un peu plus de chances de faire plus de victimes qu'un accident urbain.

**Conclusion :** les deux tests **confirment le J3**. La nuit sans éclairage et la campagne rendent les accidents plus graves, et ce n'est pas dû au hasard. Les effets sont **réels mais modérés** : ces facteurs augmentent le risque sans tout expliquer.


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [10]:
from sklearn.metrics import silhouette_score

GOLD_SEGMENTS_PATH = "/lakehouse/gold/segments_districts"
FEATURES = ["nb_accidents", "pct_graves_mortels", "pct_nuit", "pct_rural", "pct_vitesse_60_plus"]

# 1) — Agréger les features par district CÔTÉ SPARK (2 M lignes → ~416 lignes)

def pct(condition):
    """% de lignes qui respectent la condition (les NULL comptent comme 0)."""
    return F.round(F.avg(F.when(condition, 1).otherwise(0)) * 100, 2)

entities_pd = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .filter(F.col("local_authority_district").isNotNull())
    .groupBy(F.col("local_authority_district").alias("district"))
    .agg(
        F.count("*").alias("nb_accidents"),
        pct(F.col("accident_severity").isin("Serious", "Fatal")).alias("pct_graves_mortels"),
        pct(F.col("is_night")).alias("pct_nuit"),
        pct(F.col("urban_or_rural_area") == "Rural").alias("pct_rural"),
        pct(F.col("speed_limit") >= 60).alias("pct_vitesse_60_plus"),
    )
    .filter("nb_accidents >= 100")          # on écarte les districts trop petits
    .toPandas()                             # seulement ~416 lignes → sans risque
)
assert entities_pd is not None, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."
print(f"{len(entities_pd)} districts à segmenter\n")
print(entities_pd[FEATURES].describe().round(1), "\n")


# 2.a) — Prétraitement : log1p (asymétrie) + StandardScaler (même échelle)

X = entities_pd[FEATURES].copy()
X["nb_accidents"] = np.log1p(X["nb_accidents"])     # écrase les très grandes villes
X_scaled = StandardScaler().fit_transform(X)         # moyenne 0, écart-type 1

# 2.b) — Choisir k avec le score de silhouette

print("Score de silhouette selon k (plus haut = groupes mieux séparés) :")
for k in range(2, 7):
    labels_k = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    print(f"  k = {k} : {silhouette_score(X_scaled, labels_k):.3f}")

K = 3   # compromis : bon score + groupes faciles à interpréter
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled).astype(int)

# 2.c) — Interpréter : médianes par cluster (sur les valeurs D'ORIGINE)

profil = entities_pd.groupby("cluster_id")[FEATURES].median().round(1)
profil["nb_districts"] = entities_pd["cluster_id"].value_counts().sort_index()

# Nommage métier : on classe les clusters du moins rural au plus rural
NOMS = {
    3: ["Urbain dense : accidents fréquents mais peu graves",
        "Mixte / périurbain",
        "Rural : moins d'accidents mais plus graves"],
    4: ["Urbain très dense", "Urbain", "Périurbain", "Rural : accidents graves"],
}[K]
ordre = profil.sort_values("pct_rural").index
noms_clusters = {cid: NOMS[i] for i, cid in enumerate(ordre)}
entities_pd["segment"] = entities_pd["cluster_id"].map(noms_clusters)
profil["segment"] = profil.index.map(noms_clusters)

print("\n=== Profil médian de chaque segment ===")
print(profil.sort_values("pct_rural").to_string(), "\n")

print("=== Exemples de districts par segment (les plus touchés) ===")
for seg, grp in entities_pd.groupby("segment"):
    print(f"- {seg} : {', '.join(grp.nlargest(5, 'nb_accidents')['district'])}")

assert "segment" in entities_pd.columns, "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."

# 3) — Sauvegarder en Delta Lake sur HDFS (Gold)

(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"\nTable de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} entités) !")


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


416 districts à segmenter

       nb_accidents  pct_graves_mortels  pct_nuit  pct_rural  \
count         416.0               416.0     416.0      416.0   
mean         4921.3                16.2      26.2       46.6   
std          3849.9                 4.2       2.8       29.6   
min           318.0                 6.9      15.5        0.0   
25%          2704.5                13.1      24.4       19.0   
50%          4035.5                15.7      26.5       49.8   
75%          5861.2                19.0      28.0       71.4   
max         39662.0                31.0      33.6      100.0   

       pct_vitesse_60_plus  
count                416.0  
mean                  30.2  
std                   21.2  
min                    0.0  
25%                   10.7  
50%                   29.5  
75%                   46.6  
max                   78.3   

Score de silhouette selon k (plus haut = groupes mieux séparés) :
  k = 2 : 0.343
  k = 3 : 0.249
  k = 4 : 0.233
  k = 5 : 0.223
  k

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()



Table de segmentation enregistrée dans /lakehouse/gold/segments_districts (416 entités) !


#### Interprétation de la segmentation

K-Means a classé les 416 districts en **3 groupes** :

| Segment | Districts | Accidents graves | Exemples |
|---|---|---|---|
| **Urbain dense** | 96 | **13 %** | Birmingham, Leeds, Glasgow |
| **Mixte** | 153 | **15 %** | Plymouth, Milton Keynes |
| **Rural** | 167 | **19 %** | Cornwall, Wiltshire |

- **En ville :** beaucoup d'accidents, mais peu graves (on roule lentement).
- **À la campagne :** moins d'accidents, mais beaucoup plus graves (routes rapides).
- La **nuit** est à peu près la même partout (25 à 28 %) : ce n'est pas elle qui distingue les districts.

**Pourquoi k = 3 :** k = 2 a le meilleur score (0,34), mais il sépare seulement « ville » et « campagne ». k = 3 ajoute un groupe **mixte**, plus utile, avec un score encore correct (0,25).

**Conclusion :** plus un district est **rural**, plus ses accidents sont **graves**. La prévention doit viser les **routes rapides de campagne**.


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [11]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/evolution_mensuelle       :       156 lignes


 - /lakehouse/gold/facteurs_gravite          :        37 lignes


 - /lakehouse/gold/pareto_districts          :       416 lignes
 - /lakehouse/gold/segments_districts        :       416 lignes
